In [1]:
import json
import os
import re
from pathlib import Path
from typing import Optional

import numpy as np
import pandas as pd
from openai import OpenAI
from rapidfuzz import fuzz
from rapidfuzz.distance import JaroWinkler
from sentence_transformers import SentenceTransformer
from sklearn.neighbors import NearestNeighbors


def _normalize_text(value: object) -> str:
    """Normalize text for comparison."""
    if value is None or (isinstance(value, float) and np.isnan(value)):
        return ""
    text = str(value).lower()
    text = re.sub(r"[^a-z0-9\s]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text


def _extract_phone_candidates(raw: str) -> list[str]:
    if not raw:
        return []
    cleaned = raw.replace(";", ",").replace("|", ",")
    pattern = r"\+?\d[\d\-\s\(\)\./]{6,}\d"
    candidates = re.findall(pattern, cleaned)
    if not candidates:
        return [cleaned]
    return candidates


def _normalize_phone_set(value: object, default_region: str = "CA") -> set[str]:
    """Normalize phone numbers into a set to handle multiple values in one field."""
    if value is None or (isinstance(value, float) and np.isnan(value)):
        return set()
    raw = str(value).strip()
    if not raw:
        return set()

    numbers: set[str] = set()
    candidates = _extract_phone_candidates(raw)

    for candidate in candidates:
        candidate = re.split(r"(?:ext|x)\s*\d+", candidate, flags=re.IGNORECASE)[0]
        digits_only = re.sub(r"\D", "", candidate)
        if len(digits_only) < 7:
            continue
        try:
            import phonenumbers

            parsed = phonenumbers.parse(candidate, default_region)
            if phonenumbers.is_valid_number(parsed):
                numbers.add(phonenumbers.format_number(parsed, phonenumbers.PhoneNumberFormat.E164))
            else:
                numbers.add(digits_only)
        except Exception:
            numbers.add(digits_only)

    return numbers


def _column_score(a: str, b: str) -> Optional[float]:
    """Calculate similarity score between two strings."""
    if not a or not b:
        return None
    jw = JaroWinkler.similarity(a, b)
    ts = fuzz.token_set_ratio(a, b) / 100.0
    return max(jw, ts)


def _llm_judge_match(
    pairs: list[dict],
    model: Optional[str] = None,
    batch_size: int = 10,
) -> list[bool]:
    """Use LLM to judge if pairs match."""
    client = OpenAI()
    model = model or os.getenv("OPENAI_MODEL", "gpt-5-nano")
    decisions: list[bool] = []

    system_prompt = (
        "You are a matching judge. Decide if two records describe the same physical facility. "
        "Return JSON only."
    )

    for start in range(0, len(pairs), batch_size):
        batch = pairs[start:start + batch_size]
        payload = [
            {
                "generated": item["generated"],
                "ground_truth": item["ground_truth"],
            }
            for item in batch
        ]

        user_prompt = (
            "For each pair, decide if they refer to the same physical facility. "
            "If either side has missing values, use best judgment. "
            "Reply with JSON in the form {\"decisions\":[true,false,...]} in the same order.\n" 
            f"Pairs: {json.dumps(payload, ensure_ascii=True)}"
        )

        response = client.chat.completions.create(
            model=model,
            messages=[
                {"role": "system", "content": system_prompt},
                {"role": "user", "content": user_prompt},
            ],
        )

        text = response.choices[0].message.content or ""
        try:
            parsed = json.loads(text)
            batch_decisions = parsed.get("decisions", [])
        except json.JSONDecodeError:
            batch_decisions = []

        if len(batch_decisions) != len(batch):
            batch_decisions = [False] * len(batch)

        decisions.extend(bool(x) for x in batch_decisions)

    return decisions


def evaluate_dataset_matching(
    ground_truth_csv: str | Path,
    generated_csv: str | Path,
    output_dir: str | Path,
    soft_id_cols: Optional[list[str]] = None,
    hard_id_col: Optional[str] = "phone",
    default_region: str = "CA",
    vector_threshold: float = 0.80,
    auto_match_threshold: float = 0.90,
    auto_no_match_threshold: float = 0.60,
    use_llm_judge: bool = True,
) -> dict:
    """
    Evaluate generated dataset against ground truth using matching algorithm.
    
    Args:
        ground_truth_csv: Path to ground truth CSV
        generated_csv: Path to generated dataset CSV
        output_dir: Directory to save output CSVs
        soft_id_cols: Columns to use for soft matching (default: ["address", "name"])
        hard_id_col: Column for hard ID matching (default: "phone")
        default_region: Default region for phone normalization
        vector_threshold: Threshold for vector similarity blocking
        auto_match_threshold: Threshold for automatic match decision
        auto_no_match_threshold: Threshold below which pairs are definitely not matches
        use_llm_judge: Whether to use LLM for gray area decisions
        
    Returns:
        Dictionary with evaluation metrics
    """
    if soft_id_cols is None:
        soft_id_cols = ["address", "name"]
    
    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)
    
    # Load datasets
    print("📊 Loading datasets...")
    df_gt = pd.read_csv(ground_truth_csv)
    df_gen = pd.read_csv(generated_csv)
    
    print(f"Ground truth records: {len(df_gt)}")
    print(f"Generated records: {len(df_gen)}")
    
    # Normalize soft columns for both datasets
    for col in soft_id_cols:
        if col not in df_gt.columns:
            df_gt[col] = ""
        if col not in df_gen.columns:
            df_gen[col] = ""
        df_gt[f"__norm_{col}"] = df_gt[col].apply(_normalize_text)
        df_gen[f"__norm_{col}"] = df_gen[col].apply(_normalize_text)
    
    # Normalize hard ID column if provided
    hard_set_col = None
    if hard_id_col and hard_id_col in df_gt.columns and hard_id_col in df_gen.columns:
        hard_set_col = f"__phone_set_{hard_id_col}"
        df_gt[hard_set_col] = df_gt[hard_id_col].apply(lambda v: _normalize_phone_set(v, default_region))
        df_gen[hard_set_col] = df_gen[hard_id_col].apply(lambda v: _normalize_phone_set(v, default_region))
    
    # Create combined text for embedding
    df_gt["__soft_text"] = df_gt[[f"__norm_{c}" for c in soft_id_cols]].agg(" ".join, axis=1).str.strip()
    df_gen["__soft_text"] = df_gen[[f"__norm_{c}" for c in soft_id_cols]].agg(" ".join, axis=1).str.strip()
    
    # Generate embeddings for both datasets
    print("🧮 Generating embeddings...")
    model = SentenceTransformer("all-MiniLM-L6-v2")
    
    all_texts = df_gt["__soft_text"].tolist() + df_gen["__soft_text"].tolist()
    all_embeddings = model.encode(
        all_texts,
        batch_size=64,
        show_progress_bar=True,
        normalize_embeddings=True,
    )
    all_embeddings = np.asarray(all_embeddings)
    
    gt_embeddings = all_embeddings[:len(df_gt)]
    gen_embeddings = all_embeddings[len(df_gt):]
    
    # Find candidate matches using vector similarity
    print("🔍 Finding candidate matches...")
    radius = 1.0 - vector_threshold
    nn = NearestNeighbors(metric="cosine", radius=radius)
    nn.fit(gt_embeddings)
    _, indices = nn.radius_neighbors(gen_embeddings, radius=radius)
    
    # Build candidate pairs (gen_idx, gt_idx)
    candidate_pairs: set[tuple[int, int]] = set()
    for gen_idx, gt_neighbors in enumerate(indices):
        for gt_idx in gt_neighbors:
            candidate_pairs.add((gen_idx, gt_idx))

    # Add phone-based candidates to handle multiple numbers in one field
    if hard_set_col:
        phone_to_gt: dict[str, list[int]] = {}
        for gt_idx, phone_set in enumerate(df_gt[hard_set_col]):
            for phone in phone_set:
                phone_to_gt.setdefault(phone, []).append(gt_idx)

        for gen_idx, phone_set in enumerate(df_gen[hard_set_col]):
            for phone in phone_set:
                for gt_idx in phone_to_gt.get(phone, []):
                    candidate_pairs.add((gen_idx, gt_idx))
    
    print(f"Found {len(candidate_pairs)} candidate pairs")
    
    # Score candidate pairs using fuzzy matching
    print("📊 Scoring candidate pairs...")
    confirmed_matches: set[tuple[int, int]] = set()
    gray_pairs: list[tuple[int, int]] = []
    
    for gen_idx, gt_idx in candidate_pairs:
        # Phone overlap is a strong signal of a match
        if hard_set_col:
            gen_phones = df_gen.at[gen_idx, hard_set_col]
            gt_phones = df_gt.at[gt_idx, hard_set_col]
            if gen_phones and gt_phones and gen_phones.intersection(gt_phones):
                confirmed_matches.add((gen_idx, gt_idx))
                continue

        # Skip if either has no soft text
        if not df_gen.at[gen_idx, "__soft_text"] or not df_gt.at[gt_idx, "__soft_text"]:
            gray_pairs.append((gen_idx, gt_idx))
            continue
        
        # Calculate scores for each column
        scores = []
        missing_data = False
        for col in soft_id_cols:
            a = df_gen.at[gen_idx, f"__norm_{col}"]
            b = df_gt.at[gt_idx, f"__norm_{col}"]
            score = _column_score(a, b)
            if score is None:
                missing_data = True
                break
            scores.append(score)
        
        if missing_data or not scores:
            gray_pairs.append((gen_idx, gt_idx))
            continue
        
        # Take minimum score (most conservative)
        final_score = min(scores)
        
        if final_score >= auto_match_threshold:
            confirmed_matches.add((gen_idx, gt_idx))
        elif final_score < auto_no_match_threshold:
            # Definitely not a match
            continue
        else:
            # Gray area - needs LLM judgment
            gray_pairs.append((gen_idx, gt_idx))
    
    print(f"Auto-confirmed matches: {len(confirmed_matches)}")
    print(f"Gray area pairs: {len(gray_pairs)}")
    
    # Use LLM judge for gray area pairs
    if gray_pairs and use_llm_judge:
        print("🤖 Using LLM judge for gray area pairs...")
        pair_payload = []
        for gen_idx, gt_idx in gray_pairs:
            cols_to_include = soft_id_cols + ([hard_id_col] if hard_id_col else [])
            pair_payload.append({
                "generated": df_gen.loc[gen_idx, cols_to_include].to_dict(),
                "ground_truth": df_gt.loc[gt_idx, cols_to_include].to_dict(),
                "index_pair": (gen_idx, gt_idx),
            })
        
        decisions = _llm_judge_match(pair_payload)
        for payload, is_match in zip(pair_payload, decisions):
            if is_match:
                confirmed_matches.add(payload["index_pair"])
    
    # Build match mapping: gen_idx -> gt_idx
    # Each generated record can match at most one ground truth record (take best match)
    gen_to_gt_match: dict[int, int] = {}
    matched_gt_indices: set[int] = set()
    
    for gen_idx, gt_idx in confirmed_matches:
        # If this generated record already has a match, skip
        # (In a more sophisticated version, we could keep the best match)
        if gen_idx not in gen_to_gt_match:
            gen_to_gt_match[gen_idx] = gt_idx
            matched_gt_indices.add(gt_idx)
    
    # Calculate metrics
    print("\n📈 Calculating metrics...")
    
    # True Positives: generated records that matched ground truth
    tp_indices = list(gen_to_gt_match.keys())
    true_positives = len(tp_indices)
    
    # False Positives: generated records that didn't match any ground truth
    fp_indices = [i for i in range(len(df_gen)) if i not in gen_to_gt_match]
    false_positives = len(fp_indices)
    
    # False Negatives: ground truth records that weren't matched by any generated record
    fn_indices = [i for i in range(len(df_gt)) if i not in matched_gt_indices]
    false_negatives = len(fn_indices)
    
    # Calculate precision, recall, F1
    precision = true_positives / (true_positives + false_positives) if (true_positives + false_positives) > 0 else 0
    recall = true_positives / (true_positives + false_negatives) if (true_positives + false_negatives) > 0 else 0
    f1 = 2 * (precision * recall) / (precision + recall) if (precision + recall) > 0 else 0
    
    # Save True Positives CSV
    df_tp = df_gen.iloc[tp_indices].drop(columns=[c for c in df_gen.columns if c.startswith("__")])
    tp_output_path = output_dir / "true_positives.csv"
    df_tp.to_csv(tp_output_path, index=False)
    
    # Save False Positives CSV
    df_fp = df_gen.iloc[fp_indices].drop(columns=[c for c in df_gen.columns if c.startswith("__")])
    fp_output_path = output_dir / "false_positives.csv"
    df_fp.to_csv(fp_output_path, index=False)
    
    # Save False Negatives CSV (from ground truth)
    df_fn = df_gt.iloc[fn_indices].drop(columns=[c for c in df_gt.columns if c.startswith("__")])
    fn_output_path = output_dir / "false_negatives.csv"
    df_fn.to_csv(fn_output_path, index=False)
    
    # Print results
    print("\n" + "="*60)
    print("📊 EVALUATION RESULTS")
    print("="*60)
    print(f"Ground Truth Records:     {len(df_gt)}")
    print(f"Generated Records:        {len(df_gen)}")
    print("-"*60)
    print(f"True Positives (TP):      {true_positives}")
    print(f"False Positives (FP):     {false_positives}")
    print(f"False Negatives (FN):     {false_negatives}")
    print("-"*60)
    print(f"Precision:                {precision:.4f} ({precision*100:.2f}%)")
    print(f"Recall:                   {recall:.4f} ({recall*100:.2f}%)")
    print(f"F1 Score:                 {f1:.4f}")
    print("="*60)
    print("\n💾 Output files saved:")
    print(f"   True Positives:  {tp_output_path}")
    print(f"   False Positives: {fp_output_path}")
    print(f"   False Negatives: {fn_output_path}")
    
    return {
        "ground_truth_count": len(df_gt),
        "generated_count": len(df_gen),
        "true_positives": true_positives,
        "false_positives": false_positives,
        "false_negatives": false_negatives,
        "precision": precision,
        "recall": recall,
        "f1_score": f1,
        "output_files": {
            "true_positives": str(tp_output_path),
            "false_positives": str(fp_output_path),
            "false_negatives": str(fn_output_path),
        }
    }

In [3]:
# Evaluate a generated dataset
results = evaluate_dataset_matching(
    ground_truth_csv="ground_truth.csv",
    generated_csv="generated_dataset.csv",
    output_dir="eval_csvs",
    soft_id_cols=["address", "name"],
    hard_id_col="phone",
    vector_threshold=0.80,
    auto_match_threshold=0.90,
    auto_no_match_threshold=0.60,
    use_llm_judge=True,
)

📊 Loading datasets...
Ground truth records: 1836
Generated records: 179
🧮 Generating embeddings...


Batches:   0%|          | 0/32 [00:00<?, ?it/s]

🔍 Finding candidate matches...
Found 0 candidate pairs
📊 Scoring candidate pairs...
Auto-confirmed matches: 0
Gray area pairs: 0

📈 Calculating metrics...

📊 EVALUATION RESULTS
Ground Truth Records:     1836
Generated Records:        179
------------------------------------------------------------
True Positives (TP):      0
False Positives (FP):     179
False Negatives (FN):     1836
------------------------------------------------------------
Precision:                0.0000 (0.00%)
Recall:                   0.0000 (0.00%)
F1 Score:                 0.0000

💾 Output files saved:
   True Positives:  eval_csvs/true_positives.csv
   False Positives: eval_csvs/false_positives.csv
   False Negatives: eval_csvs/false_negatives.csv
